In [ ]:
## Hannah's Work
## Random Forest
## Here’s a cleaner and more modular version of your code that:

    #Groups repetitive logic into concise functions
    #Uses dictionaries to handle variable names more cleanly
    #Outputs only the NetCDF file (not CSV)
    #Keeps analysis and plotting code separate
    #Reduces clutter while retaining all functionality

!pip install pyspedas
import pyspedas
from pytplot import get_data
import pandas as pd
import numpy as np
import xarray as xr
from datetime import timedelta
import os

# === Config ===
t1 = '2016-03-06/00:24:00'
t2 = '2016-03-06/00:34:00'
probes = [1]
buffer = 5  # minutes

# === Derived Setup ===
t1_ext = (pd.to_datetime(t1) - timedelta(minutes=buffer)).strftime('%Y-%m-%d/%H:%M:%S')
t2_ext = (pd.to_datetime(t2) + timedelta(minutes=buffer)).strftime('%Y-%m-%d/%H:%M:%S')
t1_dt = pd.to_datetime(t1).tz_localize('UTC')
t2_dt = pd.to_datetime(t2).tz_localize('UTC')
tag = t1.replace(":", "_").replace("/", "_") + "_to_" + t2.split("/")[1].replace(":", "_")
folder = f"data_{tag}"
os.makedirs(folder, exist_ok=True)

# === Helpers ===
def to_datetime_ns(arr): return pd.to_datetime(arr, unit='ns', utc=True)

def interp(original_time, original_data, target_time):
    original_sec = original_time.astype(int) / 1e9
    target_sec = target_time.astype(int) / 1e9
    if original_data.ndim == 1:
        return np.interp(target_sec, original_sec, original_data)
    return np.stack([np.interp(target_sec, original_sec, original_data[:, i]) for i in range(original_data.shape[1])], axis=1)

# === Data Variables to Load ===
mec_vars = ['r_gse', 'mlat', 'l_dipole', 'mlt', 'dst', 'kp']
var_labels = {
    'fgm_b_gse_srvy_l2_btot': 'Bt',
    'fgm_b_gse_srvy_l2': 'B_vec',
    'des_numberdensity_fast': 'Density',
    **{f'mec_{v}': v.upper() for v in mec_vars},
    'dsp_epsd_x': 'EPSD'
}

# === Loop ===
for probe in probes:
    # Load data
    pyspedas.mms.fgm(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.fpi(probe=probe, datatype=['des-moms'], center_measurement=True, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.mec(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.dsp(probe=probe, datatype=['epsd'], data_rate='fast', level='l2', trange=[t1_ext, t2_ext], time_clip=True)

    # Fetch and organize data
    raw_data = {k: get_data(f'mms{probe}_{k}', xarray=True) for k in var_labels}
    time_base = to_datetime_ns(raw_data['fgm_b_gse_srvy_l2_btot'].time.data)

    # Interpolation
    data_interp = {
        'Bt': raw_data['fgm_b_gse_srvy_l2_btot'].data,
        'B_vec': interp(to_datetime_ns(raw_data['fgm_b_gse_srvy_l2'].time.data), raw_data['fgm_b_gse_srvy_l2'].data, time_base),
        'Density': interp(to_datetime_ns(raw_data['des_numberdensity_fast'].time.data), raw_data['des_numberdensity_fast'].data, time_base)
    }

    for var in mec_vars:
        key = f'mec_{var}'
        data_interp[var.upper()] = interp(to_datetime_ns(raw_data[key].time.data), raw_data[key].data, time_base)

    # EPSD
    epsd_raw = raw_data['dsp_epsd_x']
    if epsd_raw is not None:
        epsd_time = to_datetime_ns(epsd_raw.time.data)
        freq = epsd_raw.v.data
        power = epsd_raw.data
        epsd_interp = np.stack([np.interp(time_base.astype(int)/1e9, epsd_time.astype(int)/1e9, power[:, i]) for i in range(len(freq))], axis=1)

        # Filter EPSD to 50–800 Hz
        freq_mask = (freq >= 50) & (freq <= 800)
        freq = freq[freq_mask]
        epsd_interp = epsd_interp[:, freq_mask]
    else:
        print(f"No EPSD data for MMS{probe}")
        freq = np.array([])
        epsd_interp = np.empty((len(time_base), 0))

    # Mask to user window
    mask = (time_base >= t1_dt) & (time_base <= t2_dt)
    time_final = time_base[mask]
    rel_time = (time_final - time_final[0]).total_seconds()

    # Create Dataset
    ds = xr.Dataset(
        {
            'Bt': ('time', data_interp['Bt'][mask]),
            'Bx': ('time', data_interp['B_vec'][mask, 0]),
            'By': ('time', data_interp['B_vec'][mask, 1]),
            'Bz': ('time', data_interp['B_vec'][mask, 2]),
            'Density': ('time', data_interp['Density'][mask]),
            'Rx': ('time', data_interp['R_GSE'][mask, 0]),
            'Ry': ('time', data_interp['R_GSE'][mask, 1]),
            'Rz': ('time', data_interp['R_GSE'][mask, 2]),
            'MLAT': ('time', data_interp['MLAT'][mask]),
            'L_DIPOLE': ('time', data_interp['L_DIPOLE'][mask]),
            'MLT': ('time', data_interp['MLT'][mask]),
            'DST': ('time', data_interp['DST'][mask]),
            'KP': ('time', data_interp['KP'][mask]),
            'EPSD': (['time', 'frequency'], epsd_interp[mask])
        },
        coords={
            'time': ('time', time_final),
            'frequency': ('frequency', freq),
            'RelativeTime': ('time', rel_time)
        },
        attrs={
            'description': f'MMS{probe} dataset with EPSD',
            'source': 'pySPEDAS and MMS L2'
        }
    )

    # Save
    out_file = f"{folder}/mms{probe}_multidimensional_{tag}.nc"
    ds.to_netcdf(out_file)
    print(f"Saved: {out_file}")


Saved: data_2016-03-06_00_24_00_to_00_34_00/mms1_multidimensional_2016-03-06_00_24_00_to_00_34_00.nc


In [ ]:
import os, joblib, numpy as np, pandas as pd, xarray as xr
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV


# Configuration
base_dir = "/Users/hannahw.owner/MMS Stuff/Machine learning"
model_path = os.path.join(base_dir, "ducting_detector_model.pkl")
scaler_path = os.path.join(base_dir, "scaler_RandomForestClassifier1.pkl")
data_file = os.path.join(base_dir, "DATA_week/mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc")
sample_folder = os.path.join(base_dir, "All_samples")
window_size, step_size, threshold = 160, 40, 0.42

# Helper functions
def extract_features(ds):
    df = pd.DataFrame({v: ds[v].values for v in ["Bt", "Density", "MLT", "MLat", "Dst", "Kp"]})
    for i in range(ds["EPSD"].shape[1]):
        df[f"EPSD_f{i}"] = ds["EPSD"].values[:, i]
    return df

def parse_filename_time(f):
    parts = f.replace(".nc", "").split('_')
    d = parts[2]
    s = pd.Timestamp(d) + pd.to_timedelta(f"{parts[3]}:{parts[4]}:{parts[5]}")
    e = pd.Timestamp(d) + pd.to_timedelta(f"{parts[7]}:{parts[8]}:{parts[9]}")
    return s, e

def label_window(t, intervals, duration=10):
    t_end = t + pd.Timedelta(seconds=duration)
    for istart, iend in intervals:
        if istart <= t_end and iend >= t:
            return 1
    return 0

# === Training set generation (exclude 2016-03-07 samples) ===
labeled_intervals = []
for f in os.listdir(sample_folder):
    if f.endswith(".nc") and "2016-03-07" not in f:
        start, end = parse_filename_time(f)
        labeled_intervals.append((start, end))

X, y = [], []
data_folder = os.path.join(base_dir, "DATA_week")
for file in os.listdir(data_folder):
    if not file.endswith(".nc") or "2016-03-07" in file:
        continue
    ds = xr.open_dataset(os.path.join(data_folder, file))
    times = pd.to_datetime(ds["time"].values)
    features = extract_features(ds)
    for i in range(0, len(features) - window_size, step_size):
        window = features.iloc[i:i + window_size]
        if window.isnull().values.any(): continue
        X.append(window.mean().values)
        y.append(label_window(times[i], labeled_intervals))

X = np.array(X)
y = np.array(y)

# Train model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y if len(np.unique(y)) > 1 else None
)
# Hyperparameter tuning
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': ['balanced', None]
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)
model = search.best_estimator_
print("Best hyperparameters:", search.best_params_)


# Save model and scaler
joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

# === Evaluation on 2016-03-07 ===
ds = xr.open_dataset(data_file)
features_eval = extract_features(ds)
times_eval = pd.to_datetime(ds["time"].values)
ground_truth = [parse_filename_time(f) for f in os.listdir(sample_folder) if f.endswith(".nc") and "2016-03-07" in f]

predicted = []
for i in range(0, len(features_eval) - window_size, step_size):
    flat = features_eval.iloc[i:i + window_size].mean().values.reshape(1, -1)
    if np.isnan(flat).any(): continue
    prob = model.predict_proba(scaler.transform(flat))[0][1]
    if prob >= threshold:
        predicted.append((times_eval[i], times_eval[i + window_size - 1]))

# Match predictions to ground truth
def overlap(a, b): return not (a[1] < b[0] or a[0] > b[1])

y_true, y_pred = [], []
for gt in ground_truth:
    match = any(overlap(gt, pred) for pred in predicted)
    y_true.append(1)
    y_pred.append(1 if match else 0)
for i, pred in enumerate(predicted):
    if not any(overlap(pred, gt) for gt in ground_truth):
        y_true.append(0)
        y_pred.append(1)

# Report metrics
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")


In [ ]:
# === Evaluation on 2016-03-07 ===
ds = xr.open_dataset(data_file)
features_eval = extract_features(ds)
times_eval = pd.to_datetime(ds["time"].values)
ground_truth = [parse_filename_time(f) for f in os.listdir(sample_folder) if f.endswith(".nc") and "2016-03-07" in f]

predicted = []
for i in range(0, len(features_eval) - window_size, step_size):
    flat = features_eval.iloc[i:i + window_size].mean().values.reshape(1, -1)
    if np.isnan(flat).any(): continue
    prob = model.predict_proba(scaler.transform(flat))[0][1]
    if prob >= 0.3:  # <-- Use chosen threshold here
        predicted.append((times_eval[i], times_eval[i + window_size - 1]))

# Match predictions to ground truth
def overlap(a, b): return not (a[1] < b[0] or a[0] > b[1])

y_true, y_pred = [], []
for gt in ground_truth:
    match = any(overlap(gt, pred) for pred in predicted)
    y_true.append(1)
    y_pred.append(1 if match else 0)
for i, pred in enumerate(predicted):
    if not any(overlap(pred, gt) for gt in ground_truth):
        y_true.append(0)
        y_pred.append(1)

# Report metrics
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Threshold = 0.30")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")
